# API — Akuisisi Data Entitas Budaya & Museum Bali via REST API Wikidata
**Nama:** Yuliana Farida | **NIM:** 2501010372 | **Email:** faridayuli0903@gmail.com

In [1]:
# ==============================================================================
# SOAL 3: AKUISISI DATA VIA REST API WIKIDATA (100% SEKTOR BUDAYA BALI)
# ==============================================================================
WILAYAH_BALI = [
    "Kabupaten Badung", "Kabupaten Bangli", "Kabupaten Buleleng",
    "Kabupaten Gianyar", "Kabupaten Jembrana", "Kabupaten Karangasem",
    "Kabupaten Klungkung", "Kabupaten Tabanan", "Kota Denpasar"
]

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
hasil_api_budaya = []

for wil in WILAYAH_BALI:
    nama_bersih = wil.replace("Kabupaten ", "").replace("Kota ", "").strip()
    params = {
        "action": "query",
        "list": "search",
        "srsearch": f"museum OR pura OR budaya {nama_bersih} Bali",
        "srlimit": 50,
        "format": "json"
    }
    resp = request_etis(WIKIDATA_API, params=params)
    data_json = resp.json()

    raw_json_path = RAW / f"wikidata_budaya_{nama_bersih.lower()}.json"
    raw_json_path.write_text(json.dumps(data_json, ensure_ascii=False, indent=2), encoding="utf-8")

    total_entitas = data_json.get("query", {}).get("searchinfo", {}).get("totalhits", 0)
    hasil_api_budaya.append({
        "wilayah": wil,
        "kunci_wilayah": nama_bersih.lower(),
        "jumlah_entitas_wikidata": total_entitas
    })

df_api_budaya = pd.DataFrame(hasil_api_budaya)
path_api_proc = PROC / "api_wikidata_budaya_bali.csv"
df_api_budaya.to_csv(path_api_proc, index=False)

catat_provenance(
    "API_Wikidata_Budaya_Bali",
    WIKIDATA_API,
    path_api_proc,
    "CC0 1.0 Public Domain (Wikidata Action API)",
    "REST API Query"
)

print(f"Total wilayah hasil API Wikidata Budaya: {len(df_api_budaya)} baris")
display(df_api_budaya)

In [1]:
# SOAL 2: Klasifikasi kolom PII / quasi-identifier dan Data Ethics Checklist 9 butir (Tanpa Wikipedia)
PII_PATTERN = r"nik|nama_pemilik|email|telp|phone|alamat_pribadi|ktp|npwp"
QUASI_PATTERN = r"lat|lon|latitude|longitude|kode_?pos|kecamatan|desa|kabupaten|kota|tahun|date"

df_pii = pd.DataFrame([
    {
        "kolom": c,
        "kategori": "PII Langsung" if re.search(PII_PATTERN, c, re.I)
                    else ("Quasi-Identifier" if re.search(QUASI_PATTERN, c, re.I) else "Non-PII")
    }
    for c in df_raw.columns
])
df_pii.to_csv(DOCS / "klasifikasi_pii.csv", index=False)

checklist_9 = pd.DataFrame({
    "no": range(1, 10),
    "butir_etika": [
        "1. Sumber data dan URL dicantumkan",
        "2. Lisensi data diperiksa dan dicatat",
        "3. robots.txt / Terms of Service diperiksa",
        "4. Kolom PII / quasi-identifier diidentifikasi",
        "5. PII dihapus, di-mask, atau di-pseudonimisasi",
        "6. Prinsip minimalisasi data diterapkan",
        "7. Potensi bias sampel dianalisis",
        "8. Tujuan penggunaan data dinyatakan eksplisit",
        "9. Rencana retensi dan penghapusan data ada"
    ],
    "status": ["Ya"] * 9,
    "bukti_implementasi": [
        "Tercatat lengkap pada docs/provenance.csv beserta hash MD5",
        "Lisensi Open Data Publik, CC BY 4.0 (Open-Meteo), dan Data Publik Resmi Instansi Pemerintah",
        "Bukti pengecekan robots.txt tersimpan pada docs/robots_check.json",
        "Hasil klasifikasi setiap kolom tersimpan pada docs/klasifikasi_pii.csv",
        "Kolom alamat/koordinat rinci dihapus sebelum proses penggabungan data",
        "Hanya mengambil kolom wilayah dan agregasi jumlah objek budaya",
        "Bias pencatatan pada situs budaya/museum yang terdaftar resmi",
        "Digunakan murni untuk tugas UTS sektor Budaya (non-komersial)",
        "Data mentah dihapus dari penyimpanan lokal setelah evaluasi semester selesai"
    ]
})
checklist_9.to_csv(DOCS / "data_ethics_checklist.csv", index=False)
display(checklist_9)